# Notebook 02 — Data Preprocessing Pipeline

**Project:** Cognitive Fire Defense Pipeline — AIN7601  
**Purpose:** Resize, split, convert formats, augment — output ready-to-train datasets

| Dataset | Input | Output Format | Output Size |
|---------|-------|---------------|-------------|
| A (Drone) | YOLO TXT | YOLO TXT + COCO JSON | 512×512 |
| B (Watchtower) | Any | COCO JSON | 640×640 (RT-DETR) / 800×800 (DINO) |

In [ ]:
# ── Install (Colab only) ──────────────────────────────────────────────────
# !pip install albumentations pycocotools --quiet

In [ ]:
import os, json, shutil, random
from pathlib import Path
import numpy as np
import cv2
import albumentations as A
from tqdm import tqdm

ROOT   = Path("..")
DATA_A = ROOT / "data" / "dataset-a"
DATA_B = ROOT / "data" / "dataset-b"
SEED   = 42
random.seed(SEED)
np.random.seed(SEED)

CLASSES = ["fire", "smoke"]

In [ ]:
# ── Augmentation pipeline ─────────────────────────────────────────────────
augment_train = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.RandomFog(fog_coef_range=(0.1, 0.3), p=0.3),   # simulate haze
    A.ColorJitter(hue=0.1, saturation=0.2, p=0.4),
    A.GaussNoise(var_limit=(10, 50), p=0.2),
], bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"]))

print("Augmentation pipeline ready.")

In [ ]:
# ── Dataset A preprocessing ───────────────────────────────────────────────
# Resize to 512×512, keep YOLO TXT, generate COCO JSON

TARGET_SIZE_A = (512, 512)

def resize_and_copy(src_img_dir, src_lbl_dir, dst_img_dir, dst_lbl_dir, size):
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)
    img_paths = list(src_img_dir.glob("*.jpg")) + list(src_img_dir.glob("*.png"))
    for img_path in tqdm(img_paths, desc=f"Resizing → {size}"):
        img = cv2.imread(str(img_path))
        img_resized = cv2.resize(img, size)
        cv2.imwrite(str(dst_img_dir / img_path.name), img_resized)
        lbl_src = src_lbl_dir / (img_path.stem + ".txt")
        lbl_dst = dst_lbl_dir / (img_path.stem + ".txt")
        if lbl_src.exists():
            shutil.copy(lbl_src, lbl_dst)  # YOLO TXT is normalized — no resize needed
        else:
            lbl_dst.write_text("")  # empty label for background images

raw = DATA_A / "raw"
out = DATA_A / "images"

# TODO: update split names to match actual dataset folder names after download
for split in ["train", "valid"]:
    dst_split = "val" if split == "valid" else split
    resize_and_copy(
        raw / split / "images",
        raw / split / "labels",
        out / dst_split / "images",
        out / dst_split / "labels",
        TARGET_SIZE_A
    )

print("✅ Dataset A — resize complete")

In [ ]:
# ── YOLO TXT → COCO JSON converter ───────────────────────────────────────
def yolo_to_coco(img_dir, lbl_dir, output_json, classes):
    coco = {"info": {"description": "Forest Fire Dataset"},
            "categories": [{"id": i, "name": c} for i, c in enumerate(classes)],
            "images": [], "annotations": []}
    ann_id = 0
    img_paths = sorted(img_dir.glob("*.jpg")) + sorted(img_dir.glob("*.png"))
    for img_id, img_path in enumerate(img_paths):
        img = cv2.imread(str(img_path))
        h, w = img.shape[:2]
        coco["images"].append({"id": img_id, "file_name": img_path.name, "width": w, "height": h})
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        if lbl_path.exists() and lbl_path.stat().st_size > 0:
            for line in lbl_path.read_text().strip().splitlines():
                cls_id, cx, cy, bw, bh = map(float, line.split())
                x = (cx - bw/2) * w
                y = (cy - bh/2) * h
                coco["annotations"].append({
                    "id": ann_id, "image_id": img_id, "category_id": int(cls_id),
                    "bbox": [x, y, bw*w, bh*h], "area": bw*w * bh*h, "iscrowd": 0
                })
                ann_id += 1
    output_json.parent.mkdir(parents=True, exist_ok=True)
    with open(output_json, "w") as f:
        json.dump(coco, f, indent=2)
    print(f"Saved {output_json} — {len(coco['images'])} images, {len(coco['annotations'])} annotations")

ann_dir = DATA_A / "annotations"
for split in ["train", "val"]:
    img_d = DATA_A / "images" / split / "images"
    lbl_d = DATA_A / "images" / split / "labels"
    if img_d.exists():
        yolo_to_coco(img_d, lbl_d, ann_dir / f"{split}.json", CLASSES)

print("✅ Dataset A — COCO JSON generated")